# tt-mlir #8722 repro + fix 검증

TTMetal flatbuffer translator가 D2M의 `arith.constant`를 거부하는 버그(`Dialect 'arith' not found`).
런타임: **CPU 고RAM**. colab-8570-repro 노트북 구조 재사용, upstream/main HEAD(`70b7117e5`) 기준.


In [ ]:
!nproc
!free -h

In [ ]:
!apt-get update -qq
!DEBIAN_FRONTEND=noninteractive apt-get install -y -qq clang ninja-build cmake git python3.12-venv libgtest-dev libgmock-dev

In [ ]:
%cd /content
!rm -rf /content/tt-mlir
!git clone --branch colab-8722 https://github.com/alexxony/tt-mlir.git /content/tt-mlir
%cd /content/tt-mlir
!git log --oneline -3


In [ ]:
import os
os.environ["TTMLIR_TOOLCHAIN_DIR"] = "/opt/ttmlir-toolchain/"
!mkdir -p /opt/ttmlir-toolchain
!chown -R $(whoami) /opt/ttmlir-toolchain

In [ ]:
%cd /content/tt-mlir
!cmake -B env/build env -DCMAKE_C_COMPILER=clang -DCMAKE_CXX_COMPILER=clang++
!cmake --build env/build --parallel $(nproc)


In [ ]:
%cd /content/tt-mlir
!bash -c "source env/activate && cmake -G Ninja -B build -DCMAKE_BUILD_TYPE=Release -DTTMLIR_ENABLE_STABLEHLO=ON -DTTMLIR_ENABLE_RUNTIME=OFF -DTTMLIR_ENABLE_RUNTIME_TESTS=OFF -DTTMLIR_ENABLE_OPMODEL=OFF -DTTMLIR_ENABLE_BINDINGS_PYTHON=OFF -DCMAKE_BUILD_PARALLEL_LEVEL=$(nproc)"
!bash -c "source env/activate && cmake --build build --target ttmlir-opt ttmlir-translate llvm-lit -- -j$(nproc)" 2>&1 | tee /content/main_build.log | tail -150


In [ ]:
import subprocess, os
r = subprocess.run(["tail", "-n", "60", "/content/main_build.log"], capture_output=True, text=True)
print(r.stdout)
print("ttmlir-opt exists:", os.path.exists("/content/tt-mlir/build/bin/ttmlir-opt"))

In [14]:
repro_mlir = r"""module @jit_foo attributes {mhlo.num_partitions = 1 : i32, mhlo.num_replicas = 1 : i32} {
  ttcore.device_module {
    builtin.module @jit_foo attributes {mhlo.num_partitions = 1 : i32, mhlo.num_replicas = 1 : i32} {
      func.func public @main(%arg0: tensor<f64>, %arg1: tensor<f64>, %arg2: tensor<16x22xf64>, %arg3: tensor<22x18xf64>, %arg4: tensor<18x24xf64>, %arg5: tensor<16x24xf64>) -> tensor<16x24xf64> {
        %0 = "ttir.dot_general"(%arg2, %arg3) <{batch_dims_lhs = array<i64>, batch_dims_rhs = array<i64>, contract_dims_lhs = array<i64: 1>, contract_dims_rhs = array<i64: 0>}> : (tensor<16x22xf64>, tensor<22x18xf64>) -> tensor<16x18xf64>
        %1 = "ttir.reshape"(%arg0) <{shape = [1 : i32, 1 : i32]}> : (tensor<f64>) -> tensor<1x1xf64>
        %2 = "ttir.broadcast"(%1) <{broadcast_dimensions = array<i64: 16, 18>}> : (tensor<1x1xf64>) -> tensor<16x18xf64>
        %3 = "ttir.multiply"(%0, %2) : (tensor<16x18xf64>, tensor<16x18xf64>) -> tensor<16x18xf64>
        %4 = "ttir.dot_general"(%3, %arg4) <{batch_dims_lhs = array<i64>, batch_dims_rhs = array<i64>, contract_dims_lhs = array<i64: 1>, contract_dims_rhs = array<i64: 0>}> : (tensor<16x18xf64>, tensor<18x24xf64>) -> tensor<16x24xf64>
        %5 = "ttir.reshape"(%arg1) <{shape = [1 : i32, 1 : i32]}> : (tensor<f64>) -> tensor<1x1xf64>
        %6 = "ttir.broadcast"(%5) <{broadcast_dimensions = array<i64: 16, 24>}> : (tensor<1x1xf64>) -> tensor<16x24xf64>
        %7 = "ttir.multiply"(%arg5, %6) : (tensor<16x24xf64>, tensor<16x24xf64>) -> tensor<16x24xf64>
        %8 = "ttir.add"(%4, %7) : (tensor<16x24xf64>, tensor<16x24xf64>) -> tensor<16x24xf64>
        return %8 : tensor<16x24xf64>
      }
    }
  }
}
"""
with open("/content/repro_8722.mlir", "w") as f:
    f.write(repro_mlir)
print("written")


written


In [15]:
%cd /content/tt-mlir
import subprocess
r1 = subprocess.run(
    ["bash", "-c", "source env/activate && ttmlir-opt --ttir-to-ttmetal-pipeline -o /content/repro_8722.ttmetal.mlir /content/repro_8722.mlir"],
    capture_output=True, text=True, timeout=120,
)
print("=== ttmlir-opt rc:", r1.returncode, "===")
print(r1.stdout[-2000:])
print("=== stderr ===")
print(r1.stderr[-3000:])

if r1.returncode == 0:
    out = open("/content/repro_8722.ttmetal.mlir").read()
    print("\n=== arith.constant 잔존 여부:", "arith.constant" in out, "===")
    if "arith.constant" in out:
        for line in out.splitlines():
            if "arith.constant" in line:
                print(line)
else:
    print(">>> pipeline 자체가 실패 -- 이슈의 f64 repro가 이 경로로 현재도 안 통할 수 있음, 별도 축소 필요")


/content/tt-mlir
=== ttmlir-opt rc: 0 ===

=== stderr ===


=== arith.constant 잔존 여부: True ===
        %c0 = arith.constant 0 : index


In [16]:
%cd /content/tt-mlir
import subprocess
r2 = subprocess.run(
    ["bash", "-c", "source env/activate && ttmlir-translate --ttmetal-to-flatbuffer -o /content/repro_8722.ttm /content/repro_8722.ttmetal.mlir"],
    capture_output=True, text=True, timeout=60,
)
print("=== ttmlir-translate (미패치) rc:", r2.returncode, "===")
print("stdout:", r2.stdout)
print("stderr:", r2.stderr)
expected = "Dialect `arith' not found"
print()
print(">>> negative control 판정:", "PASS (에러 정확히 재현됨)" if expected in r2.stderr else "FAIL (에러 문구 불일치 -- 수동 확인 필요)")


/content/tt-mlir
=== ttmlir-translate (미패치) rc: 0 ===
stdout: 
stderr: 

>>> negative control 판정: FAIL (에러 문구 불일치 -- 수동 확인 필요)


In [17]:
%cd /content/tt-mlir
import subprocess
r3 = subprocess.run(
    ["bash", "-c", "source env/activate && build/bin/llvm-lit -v test/ttmlir/Dialect/TTIR/metal_layout_misc.mlir"],
    capture_output=True, text=True, timeout=120,
)
print("=== rc:", r3.returncode, "===")
print(r3.stdout[-3000:])
print(r3.stderr[-1000:])


/content/tt-mlir
=== rc: 127 ===

bash: line 1: build/bin/llvm-lit: No such file or directory



In [18]:
%cd /content/tt-mlir
path = "lib/Target/TTMetal/TTMetalToFlatbufferRegistration.cpp"
src = open(path).read()

assert '#include "mlir/Dialect/EmitC/IR/EmitC.h"' in src
src = src.replace(
    '#include "mlir/Dialect/EmitC/IR/EmitC.h"',
    '#include "mlir/Dialect/Arith/IR/Arith.h"\n#include "mlir/Dialect/EmitC/IR/EmitC.h"',
    1,
)
assert "mlir::emitc::EmitCDialect, mlir::memref::MemRefDialect," in src
src = src.replace(
    "mlir::emitc::EmitCDialect, mlir::memref::MemRefDialect,",
    "mlir::emitc::EmitCDialect, mlir::memref::MemRefDialect,\n"
    "                        mlir::arith::ArithDialect,",
    1,
)
open(path, "w").write(src)
print(open(path).read())


/content/tt-mlir
// SPDX-FileCopyrightText: (c) 2024 Tenstorrent AI ULC
//
// SPDX-License-Identifier: Apache-2.0

#include "ttmlir/Dialect/TTCore/IR/TTCore.h"
#include "ttmlir/Dialect/TTKernel/IR/TTKernel.h"
#include "ttmlir/Dialect/TTMetal/IR/TTMetal.h"
#include "ttmlir/Target/TTMetal/TTMetalToFlatbuffer.h"

#include "mlir/Dialect/Arith/IR/Arith.h"
#include "mlir/Dialect/Arith/IR/Arith.h"
#include "mlir/Dialect/EmitC/IR/EmitC.h"
#include "mlir/Dialect/Func/IR/FuncOps.h"
#include "mlir/Dialect/LLVMIR/LLVMDialect.h"
#include "mlir/Dialect/MemRef/IR/MemRef.h"
#include "mlir/Target/LLVMIR/Dialect/All.h"
#include "mlir/Target/LLVMIR/Export.h"
#include "mlir/Tools/mlir-translate/Translation.h"

using namespace mlir;

namespace mlir::tt::ttmetal {

void registerTTMetalToFlatbuffer() {
  TranslateFromMLIRRegistration reg(
      "ttmetal-to-flatbuffer", "translate ttmetal dialect to flatbuffer",
      [](Operation *op, llvm::raw_ostream &os) -> LogicalResult {
        return translateTTMetalT

In [19]:
%cd /content/tt-mlir
import subprocess
rb = subprocess.run(
    ["bash", "-c", "source env/activate && cmake --build build --target ttmlir-translate -- -j$(nproc)"],
    capture_output=True, text=True, timeout=600,
)
print("=== rebuild rc:", rb.returncode, "===")
print(rb.stdout[-2000:])
print(rb.stderr[-2000:])

r4 = subprocess.run(
    ["bash", "-c", "source env/activate && ttmlir-translate --ttmetal-to-flatbuffer -o /content/repro_8722_fixed.ttm /content/repro_8722.ttmetal.mlir"],
    capture_output=True, text=True, timeout=60,
)
print("=== ttmlir-translate (패치후) rc:", r4.returncode, "===")
print("stdout:", r4.stdout)
print("stderr:", r4.stderr)
print()
print(">>> fix 판정:", "PASS (rc=0, 번역 성공)" if r4.returncode == 0 else "FAIL -- 추가 조사 필요")


/content/tt-mlir
=== rebuild rc: 0 ===
[1/3] Building CXX object lib/Target/TTMetal/CMakeFiles/obj.TTMetalTargetFlatbuffer.dir/TTMetalToFlatbufferRegistration.cpp.o
[2/3] Linking CXX static library lib/libTTMetalTargetFlatbuffer.a
[3/3] Linking CXX executable bin/ttmlir-translate


=== ttmlir-translate (패치후) rc: 0 ===
stdout: 
stderr: 

>>> fix 판정: PASS (rc=0, 번역 성공)


In [20]:
%cd /content/tt-mlir
import subprocess
subprocess.run(["git", "checkout", "--", "lib/Target/TTMetal/TTMetalToFlatbufferRegistration.cpp"])
print(open("lib/Target/TTMetal/TTMetalToFlatbufferRegistration.cpp").read())

/content/tt-mlir
// SPDX-FileCopyrightText: (c) 2024 Tenstorrent AI ULC
//
// SPDX-License-Identifier: Apache-2.0

#include "ttmlir/Dialect/TTCore/IR/TTCore.h"
#include "ttmlir/Dialect/TTKernel/IR/TTKernel.h"
#include "ttmlir/Dialect/TTMetal/IR/TTMetal.h"
#include "ttmlir/Target/TTMetal/TTMetalToFlatbuffer.h"

#include "mlir/Dialect/EmitC/IR/EmitC.h"
#include "mlir/Dialect/Func/IR/FuncOps.h"
#include "mlir/Dialect/LLVMIR/LLVMDialect.h"
#include "mlir/Dialect/MemRef/IR/MemRef.h"
#include "mlir/Target/LLVMIR/Dialect/All.h"
#include "mlir/Target/LLVMIR/Export.h"
#include "mlir/Tools/mlir-translate/Translation.h"

using namespace mlir;

namespace mlir::tt::ttmetal {

void registerTTMetalToFlatbuffer() {
  TranslateFromMLIRRegistration reg(
      "ttmetal-to-flatbuffer", "translate ttmetal dialect to flatbuffer",
      [](Operation *op, llvm::raw_ostream &os) -> LogicalResult {
        return translateTTMetalToFlatbuffer(op, os);
      },
      [](DialectRegistry &registry) {
        regist

In [21]:
%cd /content/tt-mlir
import subprocess
rb = subprocess.run(["bash","-c","source env/activate && cmake --build build --target ttmlir-translate -- -j$(nproc)"], capture_output=True, text=True, timeout=300)
print(rb.stdout[-1500:]); print(rb.stderr[-1500:])
r = subprocess.run(["bash","-c","source env/activate && ttmlir-translate --ttmetal-to-flatbuffer -o /content/repro_8722_unpatched.ttm /content/repro_8722.ttmetal.mlir"], capture_output=True, text=True, timeout=60)
print("=== rc:", r.returncode, "==="); print("stderr:", r.stderr)

/content/tt-mlir
[1/3] Building CXX object lib/Target/TTMetal/CMakeFiles/obj.TTMetalTargetFlatbuffer.dir/TTMetalToFlatbufferRegistration.cpp.o
[2/3] Linking CXX static library lib/libTTMetalTargetFlatbuffer.a
[3/3] Linking CXX executable bin/ttmlir-translate


=== rc: 1 ===
stderr: /content/repro_8722.ttmetal.mlir:8:15: error: Dialect `arith' not found for custom op 'arith.constant' 
        %c0 = arith.constant 0 : index
              ^
/content/repro_8722.ttmetal.mlir:8:15: note: Available dialects: acc, arm_neon, arm_sme, arm_sve, builtin, emitc, func, gpu, llvm, memref, nvvm, omp, ptr, rocdl, spirv, ttcore, ttkernel, ttmetal, vcix, xevm ; for more info on dialect registration see https://mlir.llvm.org/getting_started/Faq/#registered-loaded-dependent-whats-up-with-dialects-management



In [22]:
import subprocess
r = subprocess.run(["grep","-n","-B3","-A6","arith.constant","/content/repro_8722.ttmetal.mlir"], capture_output=True, text=True)
print(r.stdout)

5-    builtin.module @jit_foo attributes {mhlo.num_partitions = 1 : i32, mhlo.num_replicas = 1 : i32, ttcore.system_desc = #system_desc} {
6-      ttcore.device @default_device = <workerGrid = #ttcore.grid<8x8, virt_to_physical_map = (d0, d1) -> (0, d0, d1), physical_to_virt_map = (d0, d1, d2) -> (d1, d2)>, dramGrid = #ttcore.grid<1x12>, l1Map = (d0, d1, d2)[s0] -> (0, d0, d1, d2 + s0), dramMap = (d0, d1, d2)[s0, s1, s2, s3, s4, s5, s6] -> (0, 0, (((d0 * s1) * (s2 * (s3 * s6)) + d1 * (s2 * (s3 * s6)) + d2) floordiv s4) mod 12, ((((d0 * s1) * (s2 * (s3 * s6)) + d1 * (s2 * (s3 * s6)) + d2) floordiv s4) floordiv 12) * s4 + ((d0 * s1) * (s2 * (s3 * s6)) + d1 * (s2 * (s3 * s6)) + d2) mod s4 + s5), meshShape = , chipIds = [0]>
7-      func.func public @main(%arg0: memref<f32>, %arg1: memref<f32>, %arg2: memref<16x22xf32>, %arg3: memref<22x18xf32>, %arg4: memref<18x24xf32>, %arg5: memref<16x24xf32>) -> memref<16x24xf32> attributes {tt.function_type = "forward_device"} {
8:        %c0 = arith.

In [23]:
%cd /content/tt-mlir
path = "lib/Target/TTMetal/TTMetalToFlatbufferRegistration.cpp"
src = open(path).read()
if "mlir::arith::ArithDialect" not in src:
    src = src.replace('#include "mlir/Dialect/EmitC/IR/EmitC.h"',
                       '#include "mlir/Dialect/Arith/IR/Arith.h"\n#include "mlir/Dialect/EmitC/IR/EmitC.h"', 1)
    src = src.replace("mlir::emitc::EmitCDialect, mlir::memref::MemRefDialect,",
                       "mlir::emitc::EmitCDialect, mlir::memref::MemRefDialect,\n                        mlir::arith::ArithDialect,", 1)
    open(path,"w").write(src)
print(open(path).read())

/content/tt-mlir
// SPDX-FileCopyrightText: (c) 2024 Tenstorrent AI ULC
//
// SPDX-License-Identifier: Apache-2.0

#include "ttmlir/Dialect/TTCore/IR/TTCore.h"
#include "ttmlir/Dialect/TTKernel/IR/TTKernel.h"
#include "ttmlir/Dialect/TTMetal/IR/TTMetal.h"
#include "ttmlir/Target/TTMetal/TTMetalToFlatbuffer.h"

#include "mlir/Dialect/Arith/IR/Arith.h"
#include "mlir/Dialect/EmitC/IR/EmitC.h"
#include "mlir/Dialect/Func/IR/FuncOps.h"
#include "mlir/Dialect/LLVMIR/LLVMDialect.h"
#include "mlir/Dialect/MemRef/IR/MemRef.h"
#include "mlir/Target/LLVMIR/Dialect/All.h"
#include "mlir/Target/LLVMIR/Export.h"
#include "mlir/Tools/mlir-translate/Translation.h"

using namespace mlir;

namespace mlir::tt::ttmetal {

void registerTTMetalToFlatbuffer() {
  TranslateFromMLIRRegistration reg(
      "ttmetal-to-flatbuffer", "translate ttmetal dialect to flatbuffer",
      [](Operation *op, llvm::raw_ostream &os) -> LogicalResult {
        return translateTTMetalToFlatbuffer(op, os);
      },
      [](Di

In [24]:
import subprocess
rb = subprocess.run(["bash","-c","source env/activate && cmake --build build --target ttmlir-translate -- -j$(nproc)"], capture_output=True, text=True, timeout=300)
print(rb.stdout[-1000:])
r = subprocess.run(["bash","-c","source env/activate && ttmlir-translate --ttmetal-to-flatbuffer -o /content/repro_8722_fixed.ttm /content/repro_8722.ttmetal.mlir"], capture_output=True, text=True, timeout=60)
print("rc:", r.returncode, "stderr:", r.stderr)

[1/3] Building CXX object lib/Target/TTMetal/CMakeFiles/obj.TTMetalTargetFlatbuffer.dir/TTMetalToFlatbufferRegistration.cpp.o
[2/3] Linking CXX static library lib/libTTMetalTargetFlatbuffer.a
[3/3] Linking CXX executable bin/ttmlir-translate

rc: 0 stderr: 


In [25]:
%cd /content/tt-mlir
test_content = '''// RUN: ttmlir-opt --ttir-to-ttmetal-pipeline -o %t.mlir %s
// RUN: FileCheck %s --input-file=%t.mlir --check-prefix=PIPELINE
// RUN: ttmlir-translate --ttmetal-to-flatbuffer %t.mlir > %t.ttm

// Regression test for https://github.com/tenstorrent/tt-mlir/issues/8722
// TTMetal flatbuffer translator must register the arith dialect: scalar
// arguments materialized via host-layout copies lower to arith.constant
// (e.g. index constants for scalar buffer accesses), and translation must
// not reject them.
module @jit_foo attributes {mhlo.num_partitions = 1 : i32, mhlo.num_replicas = 1 : i32} {
  ttcore.device_module {
    builtin.module @jit_foo attributes {mhlo.num_partitions = 1 : i32, mhlo.num_replicas = 1 : i32} {
      func.func public @main(%arg0: tensor<f64>, %arg1: tensor<f64>, %arg2: tensor<16x22xf64>, %arg3: tensor<22x18xf64>, %arg4: tensor<18x24xf64>, %arg5: tensor<16x24xf64>) -> tensor<16x24xf64> {
        // PIPELINE: arith.constant
        %0 = "ttir.dot_general"(%arg2, %arg3) <{batch_dims_lhs = array<i64>, batch_dims_rhs = array<i64>, contract_dims_lhs = array<i64: 1>, contract_dims_rhs = array<i64: 0>}> : (tensor<16x22xf64>, tensor<22x18xf64>) -> tensor<16x18xf64>
        %1 = "ttir.reshape"(%arg0) <{shape = [1 : i32, 1 : i32]}> : (tensor<f64>) -> tensor<1x1xf64>
        %2 = "ttir.broadcast"(%1) <{broadcast_dimensions = array<i64: 16, 18>}> : (tensor<1x1xf64>) -> tensor<16x18xf64>
        %3 = "ttir.multiply"(%0, %2) : (tensor<16x18xf64>, tensor<16x18xf64>) -> tensor<16x18xf64>
        %4 = "ttir.dot_general"(%3, %arg4) <{batch_dims_lhs = array<i64>, batch_dims_rhs = array<i64>, contract_dims_lhs = array<i64: 1>, contract_dims_rhs = array<i64: 0>}> : (tensor<16x18xf64>, tensor<18x24xf64>) -> tensor<16x24xf64>
        %5 = "ttir.reshape"(%arg1) <{shape = [1 : i32, 1 : i32]}> : (tensor<f64>) -> tensor<1x1xf64>
        %6 = "ttir.broadcast"(%5) <{broadcast_dimensions = array<i64: 16, 24>}> : (tensor<1x1xf64>) -> tensor<16x24xf64>
        %7 = "ttir.multiply"(%arg5, %6) : (tensor<16x24xf64>, tensor<16x24xf64>) -> tensor<16x24xf64>
        %8 = "ttir.add"(%4, %7) : (tensor<16x24xf64>, tensor<16x24xf64>) -> tensor<16x24xf64>
        return %8 : tensor<16x24xf64>
      }
    }
  }
}
'''
import subprocess
with open("test/ttmlir/Dialect/TTIR/metal_arith_constant_translate.mlir", "w") as f:
    f.write(test_content)
print("written")

/content/tt-mlir
written


In [26]:
import subprocess
r = subprocess.run(["bash","-c","find / -maxdepth 6 -name llvm-lit -type f 2>/dev/null"], capture_output=True, text=True)
print("llvm-lit candidates:", r.stdout)

llvm-lit candidates: /opt/ttmlir-toolchain/bin/llvm-lit



In [27]:
%cd /content/tt-mlir
import subprocess
r = subprocess.run(
    ["bash", "-c", "source env/activate && /opt/ttmlir-toolchain/bin/llvm-lit -v test/ttmlir/Dialect/TTIR/metal_arith_constant_translate.mlir"],
    capture_output=True, text=True, timeout=120,
)
print("rc:", r.returncode)
print(r.stdout[-3000:])
print(r.stderr[-2000:])

/content/tt-mlir
rc: 0
-- Testing: 1 tests, 1 workers --
PASS: TTMLIR :: ttmlir/Dialect/TTIR/metal_arith_constant_translate.mlir (1 of 1)

Testing Time: 0.19s

Total Discovered Tests: 1
  Passed: 1 (100.00%)




In [29]:
%cd /content/tt-mlir
import subprocess
subprocess.run(["git", "checkout", "--", "lib/Target/TTMetal/TTMetalToFlatbufferRegistration.cpp"])
rb = subprocess.run(["bash","-c","source env/activate && cmake --build build --target ttmlir-translate -- -j$(nproc)"], capture_output=True, text=True, timeout=300)
print(rb.stdout[-800:])

/content/tt-mlir
[1/3] Building CXX object lib/Target/TTMetal/CMakeFiles/obj.TTMetalTargetFlatbuffer.dir/TTMetalToFlatbufferRegistration.cpp.o
[2/3] Linking CXX static library lib/libTTMetalTargetFlatbuffer.a
[3/3] Linking CXX executable bin/ttmlir-translate



In [30]:
import subprocess
r = subprocess.run(
    ["bash", "-c", "source env/activate && /opt/ttmlir-toolchain/bin/llvm-lit -v test/ttmlir/Dialect/TTIR/metal_arith_constant_translate.mlir"],
    capture_output=True, text=True, timeout=120,
)
print("rc:", r.returncode)
print(r.stdout[-3000:])

rc: 1
-- Testing: 1 tests, 1 workers --
FAIL: TTMLIR :: ttmlir/Dialect/TTIR/metal_arith_constant_translate.mlir (1 of 1)
******************** TEST 'TTMLIR :: ttmlir/Dialect/TTIR/metal_arith_constant_translate.mlir' FAILED ********************
Exit Code: 1

Command Output (stdout):
--
# RUN: at line 1
/content/tt-mlir/build/bin/ttmlir-opt --ttir-to-ttmetal-pipeline -o /content/tt-mlir/build/test/ttmlir/Dialect/TTIR/Output/metal_arith_constant_translate.mlir.tmp.mlir /content/tt-mlir/test/ttmlir/Dialect/TTIR/metal_arith_constant_translate.mlir
# executed command: /content/tt-mlir/build/bin/ttmlir-opt --ttir-to-ttmetal-pipeline -o /content/tt-mlir/build/test/ttmlir/Dialect/TTIR/Output/metal_arith_constant_translate.mlir.tmp.mlir /content/tt-mlir/test/ttmlir/Dialect/TTIR/metal_arith_constant_translate.mlir
# RUN: at line 2
/opt/ttmlir-toolchain/bin/FileCheck /content/tt-mlir/test/ttmlir/Dialect/TTIR/metal_arith_constant_translate.mlir --input-file=/content/tt-mlir/build/test/ttmlir/Dialect

In [31]:
%cd /content/tt-mlir
path = "lib/Target/TTMetal/TTMetalToFlatbufferRegistration.cpp"
src = open(path).read()
if "mlir::arith::ArithDialect" not in src:
    src = src.replace('#include "mlir/Dialect/EmitC/IR/EmitC.h"',
                       '#include "mlir/Dialect/Arith/IR/Arith.h"\n#include "mlir/Dialect/EmitC/IR/EmitC.h"', 1)
    src = src.replace("mlir::emitc::EmitCDialect, mlir::memref::MemRefDialect,",
                       "mlir::emitc::EmitCDialect, mlir::memref::MemRefDialect,\n                        mlir::arith::ArithDialect,", 1)
    open(path,"w").write(src)

import subprocess
rb = subprocess.run(["bash","-c","source env/activate && cmake --build build --target ttmlir-translate -- -j$(nproc)"], capture_output=True, text=True, timeout=300)
print(rb.stdout[-500:])

r = subprocess.run(
    ["bash", "-c", "source env/activate && /opt/ttmlir-toolchain/bin/llvm-lit -v test/ttmlir/Dialect/TTIR/metal_arith_constant_translate.mlir"],
    capture_output=True, text=True, timeout=120,
)
print("rc:", r.returncode)
print(r.stdout[-3000:])

/content/tt-mlir
[1/3] Building CXX object lib/Target/TTMetal/CMakeFiles/obj.TTMetalTargetFlatbuffer.dir/TTMetalToFlatbufferRegistration.cpp.o
[2/3] Linking CXX static library lib/libTTMetalTargetFlatbuffer.a
[3/3] Linking CXX executable bin/ttmlir-translate

rc: 0
-- Testing: 1 tests, 1 workers --
PASS: TTMLIR :: ttmlir/Dialect/TTIR/metal_arith_constant_translate.mlir (1 of 1)

Testing Time: 0.14s

Total Discovered Tests: 1
  Passed: 1 (100.00%)

